# 06 - Qwen Report Validation And Latest-Metric Eval Workflow

# 06 - Qwen 报告验证与最新指标评估流程

This notebook replaces the old `05` workflow for the current polishing stage. It runs the combined Task 1 + Task 2 flow: reuse the existing Qwen LoRA adapter, generate a project report with validation/fallback, evaluate it with the latest public metrics, and back up the accepted outputs.

本 notebook 用于替代旧 `05` 流程，服务当前打磨阶段。它串起 Task 1 + Task 2：复用已有 Qwen LoRA adapter，生成带验证/兜底的项目报告，使用最新公开指标运行评估，并备份最终被接受的输出。


## Scope / 范围

- Does not retrain YOLO.  
  不重新训练 YOLO。
- Does not delete Drive artifacts.  
  不删除 Drive 产物。
- Reuses the Drive adapter when complete; incomplete adapters are detected before report generation.  
  adapter 完整时直接复用；不完整会在报告生成前被识别。
- Uses latest public metrics: box mAP50 `0.6746`, mask mAP50 `0.6712`.  
  使用最新公开指标：box mAP50 `0.6746`，mask mAP50 `0.6712`。


In [ ]:
# CN: 挂载 Google Drive，并设置统一路径变量。
# EN: Mount Google Drive and configure shared paths.
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/CarDD_YOLO11')
REPO_URL = 'https://github.com/lsdlBlueDragon/ai-powered-vehicle-damage-assessment-pipeline.git'
REPO_CLONE_DIR = Path('/content/ai-powered-vehicle-damage-assessment-pipeline')

YOLO_WEIGHTS = DRIVE_ROOT / 'runs/train/yolo11n_seg/weights/best.pt'
QWEN_ADAPTER_DIR = DRIVE_ROOT / 'llm_adapters/qwen2_5_7b_cardd_report_lora'
CONTEXT_JSON = DRIVE_ROOT / 'reports/qwen7b_report_context.json'
QWEN_REPORT_MD = DRIVE_ROOT / 'reports/qwen7b_final_report.md'
QWEN_REPORT_METADATA = DRIVE_ROOT / 'reports/qwen7b_final_report.metadata.json'
LLM_EVAL_JSON = DRIVE_ROOT / 'reports/llm_eval_summary.json'
LLM_EVAL_MD = DRIVE_ROOT / 'reports/llm_eval_summary.md'
TASK_BACKUP_ROOT = DRIVE_ROOT / 'backups'

for key, value in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'YOLO_WEIGHTS': YOLO_WEIGHTS,
    'QWEN_ADAPTER_DIR': QWEN_ADAPTER_DIR,
    'CONTEXT_JSON': CONTEXT_JSON,
    'QWEN_REPORT_MD': QWEN_REPORT_MD,
    'QWEN_REPORT_METADATA': QWEN_REPORT_METADATA,
    'LLM_EVAL_JSON': LLM_EVAL_JSON,
    'LLM_EVAL_MD': LLM_EVAL_MD,
}.items():
    os.environ[key] = str(value)

print('DRIVE_ROOT =', DRIVE_ROOT)
print('YOLO_WEIGHTS =', YOLO_WEIGHTS)
print('QWEN_ADAPTER_DIR =', QWEN_ADAPTER_DIR)


In [ ]:
# CN: 从 GitHub 同步最新轻量工程代码到 Drive。不会复制数据集、YOLO 权重、ONNX 或 adapter。
# EN: Sync latest lightweight engineering code from GitHub to Drive. Datasets, YOLO weights, ONNX, and adapters are not copied.
%cd /content
!rm -rf ai-powered-vehicle-damage-assessment-pipeline
!git clone "$REPO_URL"

!mkdir -p "$DRIVE_ROOT/src" "$DRIVE_ROOT/scripts" "$DRIVE_ROOT/notebooks" "$DRIVE_ROOT/docs" "$DRIVE_ROOT/configs" "$DRIVE_ROOT/tests"
!rsync -a /content/ai-powered-vehicle-damage-assessment-pipeline/src/ "$DRIVE_ROOT/src/"
!rsync -a /content/ai-powered-vehicle-damage-assessment-pipeline/scripts/ "$DRIVE_ROOT/scripts/"
!rsync -a /content/ai-powered-vehicle-damage-assessment-pipeline/notebooks/ "$DRIVE_ROOT/notebooks/"
!rsync -a /content/ai-powered-vehicle-damage-assessment-pipeline/docs/ "$DRIVE_ROOT/docs/"
!rsync -a /content/ai-powered-vehicle-damage-assessment-pipeline/configs/ "$DRIVE_ROOT/configs/"
!rsync -a /content/ai-powered-vehicle-damage-assessment-pipeline/tests/ "$DRIVE_ROOT/tests/"
!cp /content/ai-powered-vehicle-damage-assessment-pipeline/README.md "$DRIVE_ROOT/README.md"
!cp /content/ai-powered-vehicle-damage-assessment-pipeline/requirements-colab.txt "$DRIVE_ROOT/requirements-colab.txt"
!cp /content/ai-powered-vehicle-damage-assessment-pipeline/pyproject.toml "$DRIVE_ROOT/pyproject.toml"


In [ ]:
# CN: 安装依赖并以 editable 模式安装项目。若当前运行时已安装，可重复运行。
# EN: Install dependencies and this project in editable mode. Safe to rerun in the same runtime.
%cd /content/drive/MyDrive/CarDD_YOLO11
!apt-get update -qq
!apt-get install -y -qq git rsync
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements-colab.txt
!python -m pip install -q -e .


In [ ]:
# CN: 检查权重、adapter 和最新指标文件。这里不训练，只验证恢复条件。
# EN: Check weights, adapter, and latest metric files. This validates recovery conditions without training.
from pathlib import Path
import json

required_files = {
    'YOLO weights / YOLO 权重': YOLO_WEIGHTS,
    'Qwen adapter config / Qwen adapter 配置': QWEN_ADAPTER_DIR / 'adapter_config.json',
    'Qwen adapter weights / Qwen adapter 权重': QWEN_ADAPTER_DIR / 'adapter_model.safetensors',
    'Qwen tokenizer config / Qwen tokenizer 配置': QWEN_ADAPTER_DIR / 'tokenizer_config.json',
    'Qwen tokenizer / Qwen tokenizer': QWEN_ADAPTER_DIR / 'tokenizer.json',
    'YOLO test metrics / YOLO 测试指标': DRIVE_ROOT / 'reports/yolo11n_seg_test_metrics.json',
    'YOLO run summary / YOLO 运行摘要': DRIVE_ROOT / 'reports/yolo11n_seg_run_summary.json',
    'YOLO results csv / YOLO 训练日志': DRIVE_ROOT / 'runs/train/yolo11n_seg/results.csv',
}

missing = []
for label, path in required_files.items():
    path = Path(path)
    ok = path.exists()
    print(f'{label}:', 'OK' if ok else 'MISSING', path)
    if not ok:
        missing.append(str(path))
if missing:
    raise FileNotFoundError('Missing required artifacts: ' + json.dumps(missing, ensure_ascii=False, indent=2))

metrics = json.loads((DRIVE_ROOT / 'reports/yolo11n_seg_test_metrics.json').read_text(encoding='utf-8'))
print('\nLatest test metrics / 最新测试指标:')
print(json.dumps(metrics, ensure_ascii=False, indent=2))
assert round(float(metrics['metrics/mAP50(B)']), 4) == 0.6746
assert round(float(metrics['metrics/mAP50(M)']), 4) == 0.6712


In [ ]:
# CN: 快速验证工程代码。Task 1+2 本地测试应包含 Qwen validation fallback 和最新指标 retrieval。
# EN: Quick code verification. Task 1+2 tests cover Qwen validation fallback and latest-metric retrieval.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python -m compileall src scripts
!python -m pytest tests/test_qwen_report_workflow.py tests/test_llm_eval.py tests/test_rag_eval.py -q -p no:cacheprovider


In [ ]:
# CN: 重连安全 adapter 准备。adapter 完整时会跳过训练，只写 metadata。
# EN: Reconnect-safe adapter preparation. A complete adapter skips training and writes metadata only.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python -m vehicle_damage_pipeline.llm.finetune_report_lora --drive-root "$DRIVE_ROOT" --skip-if-complete --train-examples 1200 --eval-examples 120 --epochs 1 --max-seq-length 1024

# CN: 只有明确要重训 adapter 时才取消下一行注释。
# EN: Uncomment only if you intentionally want to retrain the adapter.
# !python -m vehicle_damage_pipeline.llm.finetune_report_lora --drive-root "$DRIVE_ROOT" --force-retrain --train-examples 1200 --eval-examples 120 --epochs 1 --max-seq-length 1024


In [ ]:
# CN: 构建最新报告 context。它会读取 Drive 中的最新 YOLO 测试指标。
# EN: Build fresh report context from the latest Drive YOLO test metrics.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python -m vehicle_damage_pipeline.report.build_context --drive-root "$DRIVE_ROOT" --output "$CONTEXT_JSON"

context = json.loads(Path(CONTEXT_JSON).read_text(encoding='utf-8'))
print(json.dumps(context['test_metrics'], ensure_ascii=False, indent=2))
assert round(float(context['test_metrics']['metrics/mAP50(B)']), 4) == 0.6746
assert round(float(context['test_metrics']['metrics/mAP50(M)']), 4) == 0.6712


In [ ]:
# CN: 默认用 Qwen adapter 生成报告；生成后自动运行 Task 1 验证，不合格会 fallback 到模板。
# EN: Generate with the Qwen adapter by default; Task 1 validation runs automatically and falls back to template when needed.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python -m vehicle_damage_pipeline.report.generate --context "$CONTEXT_JSON" --output "$QWEN_REPORT_MD" --language Chinese --backend qwen --adapter-dir "$QWEN_ADAPTER_DIR" --metadata-output "$QWEN_REPORT_METADATA"


In [ ]:
# CN: 检查 Task 1 接受/回退结果。无论 backend 是 qwen_adapter 还是 template，保存的报告都应该能进入后续 eval。
# EN: Inspect Task 1 acceptance/fallback. Whether backend is qwen_adapter or template, the saved report proceeds to eval.
metadata = json.loads(Path(QWEN_REPORT_METADATA).read_text(encoding='utf-8'))
print(json.dumps(metadata, ensure_ascii=False, indent=2))

if metadata.get('backend') == 'qwen_adapter':
    assert metadata.get('qwen_validation', {}).get('passed') is True
    print('Qwen report accepted after validation. / Qwen 报告通过验证并被接受。')
elif metadata.get('fallback_reason') == 'qwen_report_validation_failed':
    assert metadata.get('qwen_validation', {}).get('passed') is False
    print('Qwen report failed validation; template fallback saved. / Qwen 未通过验证，已保存模板兜底报告。')
elif metadata.get('fallback_reason') == 'qwen_adapter_incomplete':
    print('Adapter incomplete; template fallback saved. / Adapter 不完整，已保存模板兜底报告。')
else:
    raise RuntimeError('Unexpected report metadata: ' + json.dumps(metadata, ensure_ascii=False, indent=2))


In [ ]:
# CN: 运行 Task 2 的 RAG/LLM eval。retrieval 现在检查最新公开指标 0.6746 / 0.6712。
# EN: Run Task 2 RAG/LLM eval. Retrieval now checks latest public metrics 0.6746 / 0.6712.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python -m vehicle_damage_pipeline.eval.run_llm_eval --context "$CONTEXT_JSON" --report "$QWEN_REPORT_MD" --knowledge-root "$DRIVE_ROOT" --output-json "$LLM_EVAL_JSON" --output-markdown "$LLM_EVAL_MD"

eval_summary = json.loads(Path(LLM_EVAL_JSON).read_text(encoding='utf-8'))
print(json.dumps(eval_summary, ensure_ascii=False, indent=2)[:5000])
assert eval_summary['retrieval_eval']['checks']['box_map50']['query'] == '0.6746'
assert eval_summary['retrieval_eval']['checks']['mask_map50']['query'] == '0.6712'
assert eval_summary['passed'] is True


In [ ]:
# CN: 展示最终报告、metadata 和 eval 摘要。
# EN: Display final report, metadata, and eval summary.
from IPython.display import Markdown, display

for path in [QWEN_REPORT_MD, QWEN_REPORT_METADATA, LLM_EVAL_MD, LLM_EVAL_JSON]:
    print('\n====', path, '====')
    path = Path(path)
    if not path.exists():
        print('MISSING / 缺失')
        continue
    if path.suffix == '.md':
        display(Markdown(path.read_text(encoding='utf-8')))
    else:
        print(json.dumps(json.loads(path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2)[:5000])


In [ ]:
# CN: 备份 Task 1+2 的最终证据。不会复制原始数据集、权重、ONNX 或 adapter。
# EN: Back up final Task 1+2 evidence. Raw datasets, weights, ONNX, and adapters are not copied.
from datetime import datetime
import shutil, zipfile

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
backup_dir = TASK_BACKUP_ROOT / f'qwen_report_eval_task12_{stamp}'
backup_dir.mkdir(parents=True, exist_ok=True)

items = [
    CONTEXT_JSON,
    QWEN_REPORT_MD,
    QWEN_REPORT_METADATA,
    LLM_EVAL_JSON,
    LLM_EVAL_MD,
    DRIVE_ROOT / 'reports/qwen7b_report_sft_metadata.json',
    DRIVE_ROOT / 'reports/yolo11n_seg_test_metrics.json',
    DRIVE_ROOT / 'notebooks/06_colab_qwen_report_eval_full_workflow.ipynb',
]

manifest = {'created_at': datetime.now().isoformat(timespec='seconds'), 'entries': [], 'missing': []}
for src in items:
    src = Path(src)
    if src.exists() and src.is_file():
        dst = backup_dir / src.relative_to(DRIVE_ROOT)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        manifest['entries'].append({'source': str(src), 'backup': str(dst), 'size_bytes': src.stat().st_size})
    else:
        manifest['missing'].append(str(src))

manifest_path = backup_dir / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

zip_path = backup_dir.with_suffix('.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
    for file in backup_dir.rglob('*'):
        if file.is_file():
            zf.write(file, file.relative_to(backup_dir.parent))

print('Backup dir / 备份目录:', backup_dir)
print('Backup zip / 备份压缩包:', zip_path)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
assert manifest['missing'] == []


## How To Use / 使用建议

Run this notebook after pulling commit `Task 2` or later from GitHub. Execute cells from top to bottom through the backup cell. The final Gradio demo remains in the previous public-demo script, but this notebook is the canonical report/eval flow for Task 1+2.

在 GitHub 拉取包含 Task 2 的提交后运行本 notebook。按顺序从顶部运行到备份 cell。Gradio 展示仍使用已有 public-demo 脚本；本 notebook 是 Task 1+2 的标准报告/eval 流程。
